In [0]:
"../utils/helpers.py"

def ingest_json(spark, source_path, target_path, audit_fn):
    """Helper to ingest JSON files"""
    df = spark.read.json(source_path)/Volumes/workspace/default/medallion_data/output/bronze/
    df = audit_fn(df)
    df.write.mode("append").parquet(target_path)

def ingest_csv(spark, source_path, target_path, audit_fn):
    """Helper to ingest CSV files"""
    df = spark.read.option("header", "true").csv(source_path)
    df = audit_fn(df)
    df.write.mode("append").parquet(target_path)

def ingest_day(spark, day_folder: str):
    print(f"📥 Starting Bronze Ingestion for folder: {day_folder}")

    raw_base_path = f"/Volumes/workspace/default/medallion_data/{day_folder}"
    bronze_base_path = "/Volumes/workspace/default/medallion_data/output/bronze"

    try:
        # Ingest Customers (JSON)
        print("   -> Ingesting Customers...")
        ingest_json(
            spark,
            f"{raw_base_path}/customers",
            f"{bronze_base_path}/customers",
            add_audit_columns
        )

        # Ingest Products (JSON)
        print("   -> Ingesting Products...")
        ingest_json(
            spark,
            f"{raw_base_path}/products",
            f"{bronze_base_path}/products",
            add_audit_columns
        )

        # Ingest Orders (CSV)
        print("   -> Ingesting Orders...")
        ingest_csv(
            spark,
            f"{raw_base_path}/orders",
            f"{bronze_base_path}/orders",
            add_audit_columns
        )

        print(f"✅ Bronze Ingestion complete for {day_folder}!\n")

    except Exception as e:
        print(f"❌ ERROR: Failed to ingest Bronze data for {day_folder}.")
        print(f"Details: {str(e)}")
        raise e  

if __name__ == "__main__":
    ingest_day(spark, "raw_day1")